# 05 · Forward Propagation

**Learning objectives**
1. Write a neuron as a matrix operation $Z = XW + b$, $A = f(Z)$.
2. Track shapes through a network.
3. Show that stacked linear layers collapse into one, and that a nonlinearity prevents this.

## 1. From one neuron to a layer

For one neuron, $z = \mathbf{w}^\top\mathbf{x} + b$. A layer of $m$ neurons stacks the weights as columns of $W$ with shape $(n, m)$:

$$\mathbf{z} = \mathbf{x}W + \mathbf{b}, \qquad \mathbf{a} = f(\mathbf{z})$$

For a batch of $B$ samples (rows of $X$, shape $(B,n)$):

$$Z = XW + \mathbf{b}, \qquad A = f(Z), \qquad Z \in \mathbb{R}^{B\times m}$$

The bias is added to every row by broadcasting.

**Shape rule:** $(B,n)\cdot(n,m)=(B,m)$. The inner dimensions must match.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

X = np.array([[1.0, 2.0], [0.0, 1.0]])          # batch B=2, features n=2
W1 = np.array([[0.5, -1.0, 0.2], [1.0, 0.5, -0.5]])   # (2, 3)
b1 = np.array([0.0, 0.1, -0.1])                  # (3,)
Z1 = X @ W1 + b1
print("X shape:", X.shape, " W1 shape:", W1.shape, " Z1 shape:", Z1.shape)
print("Z1 =\n", Z1)

X shape: (2, 2)  W1 shape: (2, 3)  Z1 shape: (2, 3)
Z1 =
 [[ 2.5  0.1 -0.9]
 [ 1.   0.6 -0.6]]


## 2. A two-layer forward pass

Network $2 \to 3 \to 1$, ReLU hidden layer. Sample 1 is $(1,2)$, sample 2 is $(0,1)$.

Hand check of sample 1, first unit: $1(0.5)+2(1.0)+0 = 2.5$. The matrix product computes this for every unit and sample at once.

In [2]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

from activations import relu

W2 = np.array([[1.0], [-1.0], [0.5]])
b2 = np.array([0.2])
A1 = relu(Z1)
Z2 = A1 @ W2 + b2
print("A1 =\n", A1)
print("Z2 =\n", Z2, " shape", Z2.shape)

# Check each sample with an explicit loop
loop = []
for x in X:
    h = [max(0.0, float(x @ W1[:, j] + b1[j])) for j in range(3)]
    loop.append(float(np.dot(h, W2[:, 0]) + b2[0]))
print("loop check:", np.round(loop, 4), " matches matrix form:", bool(np.allclose(loop, Z2[:, 0])))

A1 =
 [[2.5 0.1 0. ]
 [1.  0.6 0. ]]
Z2 =
 [[2.6]
 [0.6]]  shape (2, 1)
loop check: [2.6 0.6]  matches matrix form: True


## 3. Why we need a nonlinearity

Without an activation, two layers give

$$(XW_1+\mathbf{b}_1)W_2+\mathbf{b}_2 = X(W_1W_2) + (\mathbf{b}_1W_2+\mathbf{b}_2)$$

which is a single linear layer. The next cell checks this numerically, then adds a ReLU in between.

In [3]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

x = np.array([1.0, 2.0])
W1s = np.array([[0.5, -1.0], [1.0, 0.5]]); b1s = np.array([0.1, -0.2])
W2s = np.array([[2.0, 0.0], [-1.0, 1.0]]); b2s = np.array([0.3, 0.0])
two_linear = (x @ W1s + b1s) @ W2s + b2s
Wc = W1s @ W2s; bc = b1s @ W2s + b2s
print("two linear layers:", two_linear)
print("collapsed W', b': ", x @ Wc + bc, " W' =", Wc.tolist(), " b' =", bc.tolist())
print("with ReLU between:", relu(x @ W1s + b1s) @ W2s + b2s)

two linear layers: [ 5.7 -0.2]
collapsed W', b':  [ 5.7 -0.2]  W' = [[2.0, -1.0], [1.5, 0.5]]  b' = [0.7, -0.2]
with ReLU between: [5.5 0. ]


## 4. Interpretation

- The forward pass is a fixed sequence of matrix products and activations.
- Shapes must match at every interface. Most bugs are shape bugs.
- Depth adds expressive power only when a nonlinearity sits between the linear maps.

## 5. Exercises

1. For a layer with 5 inputs and 3 outputs, what is the shape of $W$ and of $Z$ for a batch of 16?
2. Compute the output of the network above for sample 2 by hand, and compare with `Z2`.
3. What happens to the collapsed weights $W'$ if you add a third linear layer with an identity matrix?